### Imports

In [ ]:
import pandas as pd
import os
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline

### Set Paths

In [ ]:
model_path = os.path.join(os.pardir, "model_files/results")
data_path = os.path.join(os.pardir, "model_files")
image_path_png = os.path.join(data_path, "images", "png_images")
image_path_svg = os.path.join(data_path, "images", "svg_images")

In [ ]:
# Load dataset
diabetes = load_diabetes(as_frame=True)["frame"]
X = diabetes.drop(columns=["target"])
y = diabetes["target"]

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

# Train Linear Regression (on unscaled data)
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

# Train Random Forest Regressor (on unscaled data)
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
)
rf_model.fit(X_train, y_train)

# Train Ridge Regression (on scaled data)
ridge_model = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("estimator", Ridge(alpha=1.0)),
    ]
)
ridge_model.fit(X_train, y_train)

In [ ]:
from model_metrics import show_cumulative_feature_importance

show_cumulative_feature_importance(
    rf_model,
    threshold=0.80,
    image_filename=os.path.join(image_path_svg, "cumulative_feat_importance.svg"),
    tick_fontsize=14,
    label_fontsize=14,
)

In [ ]:
importance_df = show_cumulative_feature_importance(rf_model, return_df=True)
print(importance_df)

In [ ]:
top = show_cumulative_feature_importance(
    rf_model,
    threshold=0.90,
    clean_names={"bp": "Blood Pressure", "bmi": "Body Mass Index", "s6": "Serum 6"},
    return_features=True,
    image_filename=os.path.join(
        image_path_svg, "cumulative_feat_importance_top_90.svg"
    ),
    tick_fontsize=14,
    label_fontsize=14,
)
print(top)

# feed the shortlist straight into a reduced model
X_reduced = X_test[top]

In [ ]:
from model_metrics import show_feature_pareto

show_feature_pareto(
    rf_model,
    threshold=0.80,
    image_filename=os.path.join(image_path_svg, "show_pareto_thresh_80.svg"),
    tick_fontsize=14,
    label_fontsize=14,
    legend_loc="bottom",
    figsize=(10, 5),
)

In [ ]:
show_feature_pareto(
    rf_model,
    display="curve",
    threshold=0.80,
    tick_fontsize=14,
    label_fontsize=14,
    legend_loc="bottom",
    figsize=(10, 5),
    image_filename=os.path.join(image_path_svg, "show_pareto_curve_only_thresh_80.svg"),
)

In [ ]:
top = show_feature_pareto(
    rf_model,
    top_n=15,
    threshold=0.90,
    clean_names={"bp": "Blood Pressure"},
    return_features=True,
    tick_fontsize=14,
    label_fontsize=14,
    legend_loc="bottom",
    figsize=(10, 5),
    image_filename=os.path.join(image_path_svg, "show_pareto_thresh_90.svg"),
)
print(top)

In [ ]:
from model_metrics import show_cumulative_feature_performance

show_cumulative_feature_performance(
    rf_model,
    X_test,
    y_test,
    metrics=["r2", "mae", "rmse"],
    retain_threshold=0.98,
    # display="cumulative_gains",
    tick_fontsize=14,
    label_fontsize=14,
    legend_loc="bottom",
    figsize=(10, 5),
    image_filename=os.path.join(
        image_path_svg, "show_cumulative_feat_performance_thresh_98.svg"
    ),
)